# TrustOCT: A Trustworthy Retinal OCT Classification Framework
### Master All-in-One Self-Contained Notebook

This notebook contains the **entire implementation** of the TrustOCT framework directly in the cells. It includes preprocessing, dataset loading, model architectures, evidential decision heads, losses, training engines, evaluation metrics, and plotting utilities. You can execute this notebook end-to-end in Google Colab.

## 1. Environment Setup & Data Mounting

In [ ]:
# Mount Google Drive to load/save models persistently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies required for explainability and augmentation
!pip install albumentations opencv-python-headless grad-cam pyyaml tqdm matplotlib scikit-learn

## 2. Framework Implementation Code

In [ ]:
# =====================================================================
# Imports and Seeds Control
# =====================================================================
import os
import sys
import json
import random
import time
from datetime import datetime
from abc import ABC, abstractmethod
from typing import Tuple, Dict, List, Union, Optional, Callable

import numpy as np
import cv2
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2
from albumentations.core.transforms_interface import ImageOnlyTransform

# Enforce exact seed control
def enforce_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[OK] Seeds locked to: {seed}")

enforce_seeds(42)

In [ ]:
# =====================================================================
# Preprocessing & Custom Transforms
# =====================================================================
class BilateralFilter(ImageOnlyTransform):
    """Custom Albumentations wrapper for OpenCV Bilateral Filter."""
    def __init__(self, d=9, sigma_color=75.0, sigma_space=75.0, always_apply=False, p=1.0):
        super().__init__(always_apply, p)
        self.d = d
        self.sigma_color = sigma_color
        self.sigma_space = sigma_space

    def apply(self, img: np.ndarray, **params) -> np.ndarray:
        if img.dtype != np.uint8:
            img_uint8 = (img * 255.0).astype(np.uint8)
            denoised = cv2.bilateralFilter(img_uint8, self.d, self.sigma_color, self.sigma_space)
            return (denoised / 255.0).astype(np.float32)
        return cv2.bilateralFilter(img, self.d, self.sigma_color, self.sigma_space)

def get_train_transforms(resize_h=224, resize_w=224) -> A.Compose:
    return A.Compose([
        BilateralFilter(d=9, sigma_color=75.0, sigma_space=75.0, p=1.0),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.Resize(height=resize_h, width=resize_w),
        A.Rotate(limit=15, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], p=1.0),
        ToTensorV2()
    ])

def get_val_transforms(resize_h=224, resize_w=224) -> A.Compose:
    return A.Compose([
        BilateralFilter(d=9, sigma_color=75.0, sigma_space=75.0, p=1.0),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.Resize(height=resize_h, width=resize_w),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], p=1.0),
        ToTensorV2()
    ])

In [ ]:
# =====================================================================
# Dataset Loading & Verification Utilities
# =====================================================================
CLASS_NAMES = ["CNV", "DME", "DRUSEN", "NORMAL"]
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
SUPPORTED_EXTENSIONS = (".jpeg", ".jpg", ".png")

class KermanyDataset(Dataset):
    def __init__(self, base_path: str, transform: Optional[Callable] = None):
        self.base_path = base_path
        self.transform = transform
        self.filepaths = []
        self.labels = []
        
        if not os.path.exists(self.base_path):
            raise FileNotFoundError(f"Path not found: {self.base_path}")

        for class_name in CLASS_NAMES:
            class_dir = os.path.join(self.base_path, class_name)
            if not os.path.exists(class_dir):
                continue
            for filename in os.listdir(class_dir):
                file_path = os.path.join(class_dir, filename)
                if os.path.splitext(filename)[1].lower() in SUPPORTED_EXTENSIONS:
                    self.filepaths.append(file_path)
                    self.labels.append(CLASS_TO_INDEX[class_name])

    def __len__(self) -> int:
        return len(self.filepaths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        filepath = self.filepaths[idx]
        label = self.labels[idx]
        image = Image.open(filepath).convert("RGB")
        image_np = np.array(image)
        
        if self.transform:
            augmented = self.transform(image=image_np)
            image_tensor = augmented["image"]
        else:
            image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).float() / 255.0
            
        return image_tensor, label

In [ ]:
# =====================================================================
# Swappable Decision Heads (Softmax vs Evidential Head)
# =====================================================================
class SoftmaxHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int, dropout_prob: float = 0.5):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_prob)
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dropout(x)
        return self.fc(x)

class EvidentialHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int, dropout_prob: float = 0.5):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_prob)
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.dropout(x)
        logits = self.fc(x)
        evidence = F.softplus(logits)
        alpha = evidence + 1.0
        return evidence, alpha

In [ ]:
# =====================================================================
# Model Components (Backbone, Fusion, CBAM, MixStyle & TrustOCT Assembly)
# =====================================================================
class ResNet50Backbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        resnet = models.resnet50(weights=weights)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        l3 = self.layer3(x)
        l4 = self.layer4(l3)
        return l3, l4

class ChannelAttention(nn.Module):
    def __init__(self, in_planes: int, ratio: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(concat))

class CBAM(nn.Module):
    def __init__(self, gate_channels: int):
        super().__init__()
        self.channel_attention = ChannelAttention(gate_channels)
        self.spatial_attention = SpatialAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)
        return x

class MultiScaleFusion(nn.Module):
    def __init__(self, layer3_channels=1024, layer4_channels=2048, out_channels=512):
        super().__init__()
        self.proj = nn.Conv2d(layer3_channels + layer4_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, layer3_out: torch.Tensor, layer4_out: torch.Tensor) -> torch.Tensor:
        h4, w4 = layer4_out.shape[2], layer4_out.shape[3]
        l3_resized = F.interpolate(layer3_out, size=(h4, w4), mode="bilinear", align_corners=False)
        fused = torch.cat([l3_resized, layer4_out], dim=1)
        return self.relu(self.bn(self.proj(fused)))

class MixStyle(nn.Module):
    def __init__(self, p: float = 0.5, alpha: float = 0.1, eps: float = 1e-6):
        super().__init__()
        self.p = p
        self.alpha = alpha
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or random.random() > self.p:
            return x
        batch_size = x.size(0)
        mu = x.mean(dim=[2, 3], keepdim=True)
        var = x.var(dim=[2, 3], keepdim=True)
        sig = (var + self.eps).sqrt()
        x_norm = (x - mu) / sig

        perm = torch.randperm(batch_size).to(x.device)
        mu_sh = mu[perm]
        sig_sh = sig[perm]

        lmda = torch.distributions.Beta(self.alpha, self.alpha).sample((batch_size, 1, 1, 1)).to(x.device)
        mu_mixed = lmda * mu + (1.0 - lmda) * mu_sh
        sig_mixed = lmda * sig + (1.0 - lmda) * sig_sh
        return x_norm * sig_mixed + mu_mixed

class TrustOCT(nn.Module):
    def __init__(
        self,
        use_multiscale: bool = True,
        use_cbam: bool = True,
        dg_type: str = "mixstyle",
        head_type: str = "edl",
        num_classes: int = 4
    ):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.use_cbam = use_cbam
        self.dg_type = dg_type.lower()
        self.head_type = head_type.lower()

        self.backbone = ResNet50Backbone(pretrained=True)
        
        if self.dg_type == "mixstyle":
            self.dg = MixStyle(p=0.5, alpha=0.1)
        else:
            self.dg = nn.Identity()

        if self.use_multiscale:
            self.fusion = MultiScaleFusion()
            self.feature_channels = 512
        else:
            self.feature_channels = 2048

        if self.use_cbam:
            self.attention = CBAM(self.feature_channels)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        if self.head_type == "softmax":
            self.head = SoftmaxHead(self.feature_channels, num_classes)
        else:
            self.head = EvidentialHead(self.feature_channels, num_classes)

    def forward(self, x: torch.Tensor):
        l3, l4 = self.backbone(x)
        if self.dg_type == "mixstyle":
            l3 = self.dg(l3)
        features = self.fusion(l3, l4) if self.use_multiscale else l4
        if self.use_cbam:
            features = self.attention(features)
        pooled = self.pool(features)
        pooled = torch.flatten(pooled, start_dim=1)
        return self.head(pooled)

In [ ]:
# =====================================================================
# EDL Dirichlet Loss Function
# =====================================================================
def kl_divergence(alpha: torch.Tensor, num_classes: int) -> torch.Tensor:
    device = alpha.device
    beta = torch.ones((1, num_classes), device=device)
    sum_alpha = torch.sum(alpha, dim=1, keepdim=True)
    sum_beta = torch.sum(beta, dim=1, keepdim=True)
    ln_gamma_alpha = torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    ln_gamma_beta = torch.sum(torch.lgamma(beta), dim=1, keepdim=True)
    return (
        torch.lgamma(sum_alpha) - torch.lgamma(sum_beta) -
        ln_gamma_alpha + ln_gamma_beta +
        torch.sum((alpha - beta) * (torch.digamma(alpha) - torch.digamma(sum_alpha)), dim=1, keepdim=True)
    )

class EdlLoss(nn.Module):
    def __init__(self, num_classes: int = 4, annealing_epochs: int = 10):
        super().__init__()
        self.num_classes = num_classes
        self.annealing_epochs = annealing_epochs

    def forward(self, alpha: torch.Tensor, target: torch.Tensor, epoch: int) -> torch.Tensor:
        y_one_hot = F.one_hot(target, num_classes=self.num_classes).float()
        sum_alpha = torch.sum(alpha, dim=1, keepdim=True)
        prob = alpha / sum_alpha
        
        # expected classification loss
        cls_loss = torch.sum((y_one_hot - prob) ** 2, dim=1, keepdim=True) + \
                   torch.sum(prob * (1.0 - prob) / (sum_alpha + 1.0), dim=1, keepdim=True)
        cls_loss = torch.mean(cls_loss)

        alpha_hat = y_one_hot + (1.0 - y_one_hot) * alpha
        kl_val = kl_divergence(alpha_hat, self.num_classes)
        annealing_coef = min(1.0, float(epoch) / float(self.annealing_epochs))
        return cls_loss + annealing_coef * torch.mean(kl_val)

In [ ]:
# =====================================================================
# Trainer Loop Engine
# =====================================================================
class Trainer:
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        head_type: str = "edl",
        epochs: int = 30,
        lr: float = 1e-4,
        experiment_dir: str = "outputs/EXP001"
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.experiment_dir = experiment_dir
        self.head_type = head_type.lower()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=epochs)
        
        if self.head_type == "edl":
            self.criterion = EdlLoss(num_classes=4, annealing_epochs=10)
        else:
            self.criterion = nn.CrossEntropyLoss()
            
        self.scaler = torch.cuda.amp.GradScaler() if self.device.type == "cuda" else None
        self.best_loss = float("inf")

    def train_epoch(self, epoch: int) -> Tuple[float, float]:
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for images, targets in self.train_loader:
            images, targets = images.to(self.device), targets.to(self.device)
            self.optimizer.zero_grad()
            
            if self.scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = self.model(images)
                    if self.head_type == "edl":
                        _, alpha = outputs
                        loss = self.criterion(alpha, targets, epoch)
                        preds = torch.argmax(alpha, dim=1)
                    else:
                        loss = self.criterion(outputs, targets)
                        preds = torch.argmax(outputs, dim=1)
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                if self.head_type == "edl":
                    _, alpha = outputs
                    loss = self.criterion(alpha, targets, epoch)
                    preds = torch.argmax(alpha, dim=1)
                else:
                    loss = self.criterion(outputs, targets)
                    preds = torch.argmax(outputs, dim=1)
                loss.backward()
                self.optimizer.step()
                
            running_loss += loss.item() * images.size(0)
            correct += (preds == targets).sum().item()
            total += images.size(0)
            
        return running_loss / total, correct / total

    def validate(self, epoch: int) -> Tuple[float, float]:
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        
        with torch.no_grad():
            for images, targets in self.val_loader:
                images, targets = images.to(self.device), targets.to(self.device)
                outputs = self.model(images)
                if self.head_type == "edl":
                    _, alpha = outputs
                    loss = self.criterion(alpha, targets, epoch)
                    preds = torch.argmax(alpha, dim=1)
                else:
                    loss = self.criterion(outputs, targets)
                    preds = torch.argmax(outputs, dim=1)
                    
                running_loss += loss.item() * images.size(0)
                correct += (preds == targets).sum().item()
                total += images.size(0)
                
        return running_loss / total, correct / total

    def fit(self):
        print(f"Training initiated on: {self.device}")
        for epoch in range(self.epochs):
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate(epoch)
            self.scheduler.step()
            
            print(f"Epoch {epoch+1:02d}/{self.epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
            
            if val_loss < self.best_loss:
                self.best_loss = val_loss
                os.makedirs(self.experiment_dir, exist_ok=True)
                torch.save({
                    'epoch': epoch,
                    'model_state': self.model.state_dict(),
                    'best_loss': self.best_loss
                }, os.path.join(self.experiment_dir, "weights_best.pth"))

In [ ]:
# =====================================================================
# Evaluation Pipelines (ECE Calibration & Diagnostics)
# =====================================================================
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score, confusion_matrix

def calculate_metrics(y_true, y_pred, y_prob) -> dict:
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    return {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "kappa": float(kappa),
        "confusion_matrix": cm.tolist()
    }

def calculate_ece(y_true, y_prob, num_bins=10) -> float:
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    accuracies = (predictions == y_true)
    ece = 0.0
    bin_boundaries = np.linspace(0, 1, num_bins + 1)
    
    for i in range(num_bins):
        lower = bin_boundaries[i]
        upper = bin_boundaries[i + 1]
        in_bin = (confidences > lower) & (confidences <= upper)
        prop = np.mean(in_bin)
        if prop > 0:
            acc_in_bin = np.mean(accuracies[in_bin])
            conf_in_bin = np.mean(confidences[in_bin])
            ece += prop * np.abs(conf_in_bin - acc_in_bin)
    return float(ece)

In [ ]:
# =====================================================================
# Explainability Utilities (Grad-CAM & LayerCAM wrapper)
# =====================================================================
from pytorch_grad_cam import GradCAM, LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

def generate_attributions(model, target_layers, image_path, target_class, output_path):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize((224, 224))
    img_np = np.array(img_resized, dtype=np.float32) / 255.0
    input_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0).to(next(model.parameters()).device)
    
    cam = LayerCAM(model=model, target_layers=target_layers)
    targets = [ClassifierOutputTarget(target_class)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
    overlay = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    Image.fromarray(overlay).save(output_path)
    print(f"[OK] Attribution saved: {output_path}")

In [ ]:
# =====================================================================
# Plotting Helpers
# =====================================================================
def plot_confusion_matrix(cm, classes, save_path):
    matrix = np.array(cm)
    plt.figure(figsize=(6, 6), dpi=150)
    plt.imshow(matrix, cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    thresh = matrix.max() / 2.
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            plt.text(j, i, format(matrix[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if matrix[i, j] > thresh else "black")
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

## 3. Dataset Download & Verification

In [ ]:
# Mount API Keys & Download
from google.colab import files
files.upload() # Upload kaggle.json key file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download Kermany dataset
!kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p datasets/Kermany

# Copy OCTID from Drive
!mkdir -p datasets/OCTID
!cp -r "/content/drive/MyDrive/OCTID/." datasets/OCTID/

## 4. Run Interactive Experiments

In [ ]:
# Configure dataset folder path
real_train_path = "datasets/Kermany/OCT2017 /train"
real_val_path = "datasets/Kermany/OCT2017 /val"

# Initialize Transforms
train_tf = get_train_transforms(224, 224)
val_tf = get_val_transforms(224, 224)

# Initialize Dataset loaders
train_dataset = KermanyDataset(real_train_path, transform=train_tf)
val_dataset = KermanyDataset(real_val_path, transform=val_tf)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# RUN EXP001: Baseline Model (Softmax, No multi-scale, No CBAM, No MixStyle)
enforce_seeds(42)
model_baseline = TrustOCT(use_multiscale=False, use_cbam=False, dg_type="identity", head_type="softmax")

trainer_baseline = Trainer(
    model=model_baseline,
    train_loader=train_loader,
    val_loader=val_loader,
    head_type="softmax",
    epochs=30,
    experiment_dir="outputs/EXP001_Baseline"
)
trainer_baseline.fit()

In [ ]:
# RUN EXP006: TrustOCT Proposed Model (EDL, Multi-scale, CBAM, MixStyle)
enforce_seeds(42)
model_proposed = TrustOCT(use_multiscale=True, use_cbam=True, dg_type="mixstyle", head_type="edl")

trainer_proposed = Trainer(
    model=model_proposed,
    train_loader=train_loader,
    val_loader=val_loader,
    head_type="edl",
    epochs=30,
    experiment_dir="outputs/EXP006_TrustOCT"
)
trainer_proposed.fit()

## 5. Diagnostics, Calibration, and Explainability

In [ ]:
# Compute validation accuracy metrics on best Proposed weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("outputs/EXP006_TrustOCT/weights_best.pth", map_location=device)
model_proposed.load_state_dict(checkpoint["model_state"])
model_proposed = model_proposed.to(device)
model_proposed.eval()

all_targets, all_preds, all_probs = [], [], []
with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        outputs = model_proposed(imgs)
        _, alpha = outputs
        probs = alpha / torch.sum(alpha, dim=1, keepdim=True)
        preds = torch.argmax(alpha, dim=1)
        
        all_targets.append(targets.numpy())
        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

y_true = np.concatenate(all_targets)
y_pred = np.concatenate(all_preds)
y_prob = np.concatenate(all_probs)

results = calculate_metrics(y_true, y_pred, y_prob)
ece = calculate_ece(y_true, y_prob)
print(f"Proposed Model Accuracy: {results['accuracy']:.4f}")
print(f"Proposed Model Macro F1: {results['f1_score']:.4f}")
print(f"Proposed Model ECE:      {ece:.4f}")

In [ ]:
# Generate Confusion Matrix plot
plot_confusion_matrix(results['confusion_matrix'], CLASS_NAMES, "outputs/EXP006_TrustOCT/confusion_matrix.png")
print("[OK] Confusion matrix plot generated successfully under outputs/EXP006_TrustOCT/")

In [ ]:
# Generate Explainability Activation Maps using LayerCAM
sample_scan = "datasets/Kermany/OCT2017 /test/NORMAL/NORMAL-1017326-1.jpeg"
if os.path.exists(sample_scan):
    target_layers = [model_proposed.backbone.layer3]
    generate_attributions(
        model=model_proposed,
        target_layers=target_layers,
        image_path=sample_scan,
        target_class=3,
        output_path="outputs/figures/explainability/layercam_proposed.png"
    )
else:
    print("Specify a valid image path to visualize CAM attributions.")

In [ ]:
# Save and zip all checkpoints, logs, and figures for download
!zip -r outputs.zip outputs/
print("[OK] Zipped all experiment folders to outputs.zip")